# V3 EDA 1 — 집 단위: 동 · 호별 공시가격, 큰 오차 거래, 등기

## 목적

V2 EDA(01~06)는 지역 · 단지 단위였다. 11 회귀 오차의 72%는 같은 면적 직전 거래가 6개월 안에 있는 거래에서 나온다 (`r11_error_share`).
남은 오차가 같은 단지 안의 **어느 동 · 어느 집**인지에서 생긴다면, 집 단위 공개 정보가 그 차이를 얼마나 담고 있는지 먼저 본다 (설계서 Phase 4-3, V3 범위 EDA 1).

1. 실거래 `아파트동명`으로 호별 공시가격을 **동 · 층 · 면적까지** 그 집에 연결할 수 있나, 연결된 값이 모델이 남긴 오차와 관련 있나
2. 오차 20% 초과 거래는 어떤 거래인가
3. 등기 여부 · 해제와 오차

## 결론 요약

- **연결은 된다.** 실거래 `아파트동명`은 2023년 계약부터 91~92% 채워져 있고(2021~22년은 0%), 동명이 있는 거래의 92~95%가 호별 공시가격의 같은 동 · 층 · 면적 칸에 연결된다.
- **하지만 공시가격은 같은 단지 안에서 동마다 거의 다르지 않다.** 같은 층 · 면적에서 동 프리미엄이 ±0.5% 안인 거래가 80.5%다.
- **모델이 남긴 오차와의 관계도 약하다.**

| 점검 (2023~25 중개거래 OOF, Test 미사용) | 결과 |
|---|---|
| 동 프리미엄으로 잔차 보정 (2023~24 추정 → 2025 적용) | 절편만 보정 4.250% → 동 프리미엄 추가 4.236% (**0.014%p**) |
| 직전 거래가 × 이번 집 / 직전 거래 집 공시 비율 | 4.976% → 층비 5.063% · 동층비 5.039% (**오히려 악화**) |
| 동이 달라 생기는 공시 차이와 잔차 | 상관 0.099, 모델이 그 차이의 약 1/3을 놓침 (−4% 구간 r +0.005 → +4% 구간 r +0.033) |

- **오차 20% 초과 거래(1.46%, 오차의 9%)는 맞힐 단서가 없는 거래가 대부분이다.**
  - 80%는 직전 비교 거래(시세 보정)보다 20% 넘게 벗어난 거래이고, 66%는 실제가 예측보다 낮다.
  - 100세대 미만(전체 7% → 이 그룹 38%), 비교 거래가 없거나 1년 넘은 거래, 신규 단지에 몰려 있다.
  - 걸러낼 만한 데이터 오류 유형은 보이지 않는다: 매도자 · 매수자가 개인이 아닌 거래는 0.6% · 0.1%, 등기 없는 거래는 0.24%다.
- **등기:** 2023-01~2025-06 계약의 99.8%가 등기를 마쳤다 (계약 → 등기 중앙 82일). 등기 없는 308건은 오차가 크지만(6.25%) 비중이 작아 처리할 필요가 없다.

**판정:** 동 · 호별 공시가격은 연결은 되지만 예상 개선이 0.01~0.05%p로 채택 기준(0.1%p)에 한참 못 미친다 → **V3 회귀 시험 후보에서 제외**한다.
큰 오차 거래는 정답(실거래가)을 보고 걸러야 해서 학습 · 평가에서 빼지 않는다. 등기 · 해제는 처리할 것이 없다.

## 원칙

- 모델 오차는 매달 재학습 OOF(`models/regression/v2/oof_predictions.parquet`, 11 보완 2 모델)의 **2023~2025 중개거래**만 본다. Test(2026)는 보지 않는다.
- 시점: Y년 공시가격은 Y년 7월 1일부터 (05b와 같은 규칙). `아파트동명`은 등기 후 공개되지만, 서비스에서는 사용자가 자기 집 동을 입력하므로 거래 대상 집의 속성으로 본다.
- 호별 공시가격 원본(`data/external/official_price/apt_official_price_seoul_*.parquet`, 약 63MB)은 커밋하지 않는다 (`05b_download_official_price.py`로 다시 받음).

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / ".env").exists():
            return path
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
PROCESSED_DIR = PROJECT_DIR / "data" / "v2_5year_seoul" / "processed"
PRICE_DIR = PROJECT_DIR / "data" / "external" / "official_price"
OOF_PATH = PROJECT_DIR / "models" / "regression" / "v2" / "oof_predictions.parquet"

## 1. 데이터 준비

- 거래: `v2_trade`(모델 입력) + `df_fe_base`(원본 열: `아파트동명` · `등기일자` · `매도자` · `매수자`) — 둘 다 `transaction_id` 234,302행
- 호별 공시가격: 필지(`lot` = 법정동코드 + 본번 + 부번) · 공시연도 · 동명 · 호명 · 전용면적
- 동 이름 정규화: 괄호 안 단지 표기와 끝의 '동'을 떼고, 숫자가 있으면 마지막 숫자 묶음(`101동` · `청운현대(아)101동` · `2단지 210` → `101` · `101` · `210`), 없으면 마지막 글자(`A동` → `A`)
- 호명 → 층: 보완 4(`r11_unit_build.py`)와 같은 규칙 ('지하' 제외, 'N층' 표기는 N, 3~4자리 숫자는 100으로 나눈 몫)

In [2]:
def parse_floor(s):
    s = str(s).strip()
    if "지하" in s:
        return np.nan
    m = re.search(r"(\d+)\s*층", s)
    if m:
        return int(m[1]) or np.nan
    ds = re.findall(r"\d+", s)
    if not ds or len(ds[-1]) not in (3, 4):
        return np.nan
    return int(ds[-1]) // 100 or np.nan


def dong_key(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    s = str(s).strip()
    if not s or s.lower() in ("none", "nan"):
        return None
    s = re.sub(r"\(.*?\)", "", s).replace("제", "").replace(" ", "")
    s = re.sub(r"동$", "", s)
    ds = re.findall(r"\d+", s)
    if ds:
        return str(int(ds[-1]))
    return s[-1:].upper() if s else None


assert [dong_key(x) for x in ["101동", "청운현대(아)101동", "2단지 210", "A동", "제3동", None, "None"]] == ["101", "101", "210", "A", "3", None, None]
assert parse_floor("1203") == 12 and parse_floor("101") == 1 and parse_floor("3층") == 3 and np.isnan(parse_floor("지하101")) and np.isnan(parse_floor("B1"))

trade = pd.read_parquet(PROCESSED_DIR / "v2_trade.parquet", columns=[
    "transaction_id", "단지일련번호", "단지명", "시군구명", "계약일자", "전용면적", "층", "최고층", "연식", "세대수", "공시연도",
    "거래유형", "거래금액", "직전거래_층", "직전거래_경과일", "직전거래가_동일면적", "직전거래가_시세보정"])
raw = pd.read_parquet(PROCESSED_DIR / "df_fe_base.parquet", columns=["transaction_id", "아파트동명", "등기일자", "매도자", "매수자", "해제여부"])
lot = pd.read_parquet(PROCESSED_DIR / "complex_attributes_v2.parquet", columns=["단지일련번호", "lot"])
trade["단지일련번호"] = trade["단지일련번호"].astype(str)
lot["단지일련번호"] = lot["단지일련번호"].astype(str)
trade = (trade.merge(raw, on="transaction_id", how="left", validate="one_to_one")
              .merge(lot, on="단지일련번호", how="left", validate="many_to_one"))
trade["계약일자"] = pd.to_datetime(trade["계약일자"])
trade["연"] = trade["계약일자"].dt.year
trade["동키"] = trade["아파트동명"].map(dong_key)
assert (trade["공시연도"].to_numpy() == np.where(trade["계약일자"].dt.month >= 7, trade["계약일자"].dt.year, trade["계약일자"].dt.year - 1)).all()
print(trade.shape, "| 해제여부:", trade["해제여부"].value_counts(dropna=False).to_dict())

lots = set(trade["lot"].dropna())
frames, notice = [], {}
for y in sorted(trade["공시연도"].unique()):
    g = pd.read_parquet(PRICE_DIR / f"apt_official_price_seoul_{y}.parquet",
                        columns=["기준연도", "공시일", "법정동코드", "본번", "부번", "동명", "호명", "전용면적", "공시가격"])
    notice[y] = pd.Timestamp(g["공시일"].iloc[0])
    g["lot"] = g["법정동코드"].astype(str).str.zfill(10) + g["본번"].astype(str).str.zfill(4) + g["부번"].astype(str).str.zfill(4)
    frames.append(g[g["lot"].isin(lots)])
unit = pd.concat(frames, ignore_index=True)
assert (trade["계약일자"] > trade["공시연도"].map(notice)).all(), "계약일 뒤에 공시된 값을 쓰면 안 됨"
unit["기준연도"] = unit["기준연도"].astype("int64")
unit["전용면적"] = pd.to_numeric(unit["전용면적"], errors="coerce").round(2)
unit["m2당"] = pd.to_numeric(unit["공시가격"], errors="coerce").astype(float) / unit["전용면적"] / 1e4  # 만원/㎡
unit = unit.dropna(subset=["m2당"])
names = pd.Series(unit["호명"].astype(str).unique())
unit["층"] = unit["호명"].astype(str).map(dict(zip(names, names.map(parse_floor))))
names = pd.Series(unit["동명"].astype(str).unique())
unit["동키"] = unit["동명"].astype(str).map(dict(zip(names, names.map(dong_key))))
print(f"호별 공시 {len(unit):,}행 | 층 읽음 {unit['층'].notna().mean():.1%} | 동 읽음 {unit['동키'].notna().mean():.1%}")
print("공시 동명 원문 예:", unit["동명"].astype(str).drop_duplicates().sample(12, random_state=1).tolist())

(234302, 25) | 해제여부: {None: 234302}


호별 공시 7,929,422행 | 층 읽음 99.9% | 동 읽음 91.6%


공시 동명 원문 예: ['425', '신영지웰 에스테이트 개봉역', '32', '1407', '경원시티아파트', '2202', '4002', '205동', '방학 지음재힐스', '534', '늘푸른아파트', '2단지 210']


## 2. 동명 채움과 연결률

거래 → 같은 필지 · 공시연도에서 가장 가까운 면적(0.5㎡ 이내, 보완 4와 같음) → 같은 층(층값) → 같은 동 · 층(동층값).

In [3]:
area_tbl = unit.groupby(["lot", "기준연도", "전용면적"])["m2당"].median().rename("면적중위").reset_index()
q = (trade[["lot", "공시연도", "전용면적"]].assign(i=np.arange(len(trade)))
     .rename(columns={"공시연도": "기준연도", "전용면적": "거래면적"}).dropna(subset=["lot"]))
q["거래면적"] = q["거래면적"].astype(float)
q["기준연도"] = q["기준연도"].astype("int64")
ref = area_tbl.sort_values("전용면적").rename(columns={"전용면적": "면적"})
ref["면적키"] = ref["면적"].astype(float)
m = (pd.merge_asof(q.sort_values("거래면적"), ref, left_on="거래면적", right_on="면적키", by=["lot", "기준연도"],
                   direction="nearest", tolerance=0.5).set_index("i").reindex(np.arange(len(trade))))
trade["면적"] = m["면적"].to_numpy()
trade["면적중위"] = m["면적중위"].to_numpy()

known = unit.dropna(subset=["층"])
floor_tbl = known.groupby(["lot", "기준연도", "전용면적", "층"])["m2당"].median().rename("v").reset_index()
dong_tbl = (known.dropna(subset=["동키"]).groupby(["lot", "기준연도", "전용면적", "동키", "층"])["m2당"]
            .agg(v="median", 호수="size", 최소="min", 최대="max").reset_index())


def lookup(table, floors, dongs=None, cols=("v",)):
    key = pd.DataFrame({"lot": trade["lot"], "기준연도": trade["공시연도"].astype("int64"), "전용면적": trade["면적"],
                        "층": np.asarray(floors, float)})
    on = ["lot", "기준연도", "전용면적", "층"]
    if dongs is not None:
        key["동키"] = np.asarray(dongs, object)
        on = on[:3] + ["동키", "층"]
    out = key.merge(table, on=on, how="left", validate="many_to_one")
    return [out[c].to_numpy() for c in cols]


(trade["층값"],) = lookup(floor_tbl, trade["층"])
trade["동층값"], trade["동층호수"], trade["동층최소"], trade["동층최대"] = lookup(dong_tbl, trade["층"], trade["동키"], ("v", "호수", "최소", "최대"))
dong_set = set(zip(unit["lot"], unit["기준연도"], unit["동키"]))
trade["동존재"] = [(l, y, d) in dong_set if d is not None else False for l, y, d in zip(trade["lot"], trade["공시연도"].astype("int64"), trade["동키"])]

has_dong = trade["동키"].notna()
print("동명 있음 (연도별):", trade.groupby("연")["동키"].apply(lambda s: s.notna().mean()).round(3).to_dict())
pd.DataFrame({"연": trade["연"], "면적 연결": trade["면적"].notna(), "공시에 같은 동 있음": trade["동존재"],
              "층값": trade["층값"].notna(), "동층값": trade["동층값"].notna()})[has_dong].groupby("연").mean().round(3)

동명 있음 (연도별): {2021: 0.0, 2022: 0.0, 2023: 0.908, 2024: 0.915, 2025: 0.921, 2026: 0.658}


,면적 연결,공시에 같은 동 있음,층값,동층값
연,,,,
2023,0.959,0.922,0.956,0.919
2024,0.976,0.958,0.973,0.955
2025,0.964,0.950,0.962,0.947
2026,0.969,0.954,0.966,0.951


## 3. 같은 단지 안에서 공시가격은 동마다 얼마나 다른가

- 같은 동 · 층 · 면적 칸 안의 호 간 차이: 칸 하나에 호가 여러 개면 라인(향)마다 값이 다를 수 있다
- 동 프리미엄 = log(동층값 / 층값): 같은 층 · 면적에서 이 동이 단지 전체 중위보다 비싼 정도

In [4]:
linked = trade["동층값"].notna()
print("동층 한 칸의 호 수:", trade.loc[linked, "동층호수"].value_counts(normalize=True).sort_index().head(6).round(3).to_dict())
spread = (trade["동층최대"] / trade["동층최소"] - 1)[linked]
print("같은 동 · 층 · 면적 안 호 간 공시 차이 (최대/최소 − 1) 중앙 / 90% / 99%:", np.round(spread.quantile([.5, .9, .99]).to_numpy(), 4))
trade["동프리미엄"] = np.log(trade["동층값"] / trade["층값"])
trade["층프리미엄"] = np.log(trade["층값"] / trade["면적중위"])
dp = trade["동프리미엄"].dropna()
print(f"동 프리미엄 |값| ≤ 0.5%: {(dp.abs() <= 0.005).mean():.1%} | 분위 1/10/50/90/99%:", np.round(dp.quantile([.01, .1, .5, .9, .99]).to_numpy(), 4))

동층 한 칸의 호 수: {1.0: 0.097, 2.0: 0.274, 3.0: 0.064, 4.0: 0.232, 5.0: 0.041, 6.0: 0.111}
같은 동 · 층 · 면적 안 호 간 공시 차이 (최대/최소 − 1) 중앙 / 90% / 99%: [0.     0.0205 0.0655]
동 프리미엄 |값| ≤ 0.5%: 80.5% | 분위 1/10/50/90/99%: [-0.0492 -0.0054  0.      0.0005  0.0464]


## 4. 동 프리미엄과 모델이 남긴 오차

OOF 잔차 `r = log(실거래 / 예측)` (APE = |1 − e^(−r)|). 모델은 상승기에 평균적으로 조금 낮게 예측해서 r 평균이 0보다 크다.
그래서 "동 정보로 보정"할 때는 **절편만 보정한 경우와 비교**해야 동 정보의 몫이 나온다 (2023~24로 추정 → 2025에 적용).

In [5]:
oof = pd.read_parquet(OOF_PATH, columns=["transaction_id", "r", "ape", "단지"])
d = trade.merge(oof, on="transaction_id", how="inner", validate="one_to_one")
d = d[d["거래유형"].eq("중개거래") & d["연"].between(2023, 2025)].copy()
assert np.allclose(d["ape"], np.abs(1 - np.exp(-d["r"])) * 100)
print(f"OOF 중개 2023~25: {len(d):,}건 | 동층값 있음 {d['동층값'].notna().mean():.1%} | r 평균 {d['r'].mean():+.4f}")

ape_of = lambda r: np.abs(1 - np.exp(-r)) * 100
z = d.dropna(subset=["동프리미엄"])
fit, test = z[z["연"] <= 2024], z[z["연"] == 2025]
coef = np.polyfit(fit["동프리미엄"], fit["r"], 1)
nonzero = test["동프리미엄"].abs() > 0.005
rows = []
for name, s in [("2025 동층값 있는 거래 전체", test), ("그중 동 프리미엄 ≠ 0", test[nonzero])]:
    rows.append({"대상": name, "n": len(s), "모델 그대로": s["ape"].mean(), "절편만 보정": ape_of(s["r"] - fit["r"].mean()).mean(),
                 "절편 + 동 프리미엄": ape_of(s["r"] - np.polyval(coef, s["동프리미엄"])).mean()})
print(f"상관(r, 동 프리미엄) {z['r'].corr(z['동프리미엄']):.3f} | 기울기 {coef[0]:.3f}")
pd.DataFrame(rows).round(3)

OOF 중개 2023~25: 158,523건 | 동층값 있음 88.4% | r 평균 +0.0149
상관(r, 동 프리미엄) 0.069 | 기울기 0.243


,대상,n,모델 그대로,절편만 보정,절편 + 동 프리미엄
0,2025 동층값 있는 거래 전체,65898,4.478,4.250,4.236
1,그중 동 프리미엄 ≠ 0,12727,4.479,4.196,4.125


## 5. 직전 거래 집과 이번 집의 공시 비율

모델의 핵심 Feature는 같은 단지 · 면적 직전 거래가(시세 보정)다. 직전 거래 집과 이번 집의 동 · 층이 다르면 그 차이만큼 보정할 수 있는지 본다.

- 직전 거래는 06과 같은 규칙으로 다시 찾는다 (직거래 제외, 같은 단지 · 면적(1㎡ 반올림), 계약일 30일 전까지) → `v2_trade.직전거래_층`과 일치하는지 확인
- 비율은 **이번 거래의 공시연도 표** 하나에서 두 집 값을 찾아 나눈다 (같은 해 공시끼리)

In [6]:
ref = trade.loc[trade["거래유형"].ne("직거래"), ["단지일련번호", "전용면적", "계약일자", "층", "동키"]].copy()
ref["면적키"] = ref["전용면적"].astype(float).round(0)
ref["사용가능일"] = ref["계약일자"] + pd.Timedelta(days=30)
ref = ref.sort_values("계약일자", kind="stable").drop_duplicates(["단지일련번호", "면적키", "사용가능일"], keep="last")
q = trade[["transaction_id", "단지일련번호", "전용면적", "계약일자"]].copy()
q["면적키"] = q["전용면적"].astype(float).round(0)
prev = pd.merge_asof(q.sort_values("계약일자"),
                     ref.rename(columns={"층": "직전층", "동키": "직전동키"})[["단지일련번호", "면적키", "사용가능일", "직전층", "직전동키"]].sort_values("사용가능일"),
                     left_on="계약일자", right_on="사용가능일", by=["단지일련번호", "면적키"], direction="backward")
trade = trade.merge(prev[["transaction_id", "직전층", "직전동키"]], on="transaction_id", validate="one_to_one")
both = trade["직전거래_층"].notna() & trade["직전층"].notna()
assert (trade.loc[both, "직전거래_층"] == trade.loc[both, "직전층"]).all(), "06 직전 거래 규칙과 다름"
print(f"직전 거래 재현: v2_trade 직전거래_층과 {both.sum():,}건 모두 일치")

(prev_floor,) = lookup(floor_tbl, trade["직전층"])
(prev_dong,) = lookup(dong_tbl, trade["직전층"], trade["직전동키"])
trade["층비"] = trade["층값"] / prev_floor
trade["동층비"] = trade["동층값"] / prev_dong

d = trade.merge(oof, on="transaction_id", how="inner", validate="one_to_one")
d = d[d["거래유형"].eq("중개거래") & d["연"].between(2023, 2025)]
w = d.dropna(subset=["직전거래가_시세보정", "층비", "동층비"]).copy()
w["동추가"] = np.log(w["동층비"]) - np.log(w["층비"])  # 층으로 설명되지 않는, 동이 달라서 생기는 공시 차이
ae = lambda p, s: (np.abs(p - s["거래금액"]) / s["거래금액"] * 100).mean()
rows = []
for name, s in [("전체", w), ("동이 달라 동층비 ≠ 층비", w[w["동추가"].abs() > 0.005])]:
    rows.append({"대상": name, "n": len(s), "직전 시세보정가": ae(s["직전거래가_시세보정"], s),
                 "× 층비": ae(s["직전거래가_시세보정"] * s["층비"], s), "× 동층비": ae(s["직전거래가_시세보정"] * s["동층비"], s), "모델 OOF": s["ape"].mean()})
display(pd.DataFrame(rows).round(3))
print(f"상관(r, 동추가) {w['r'].corr(w['동추가']):.3f}")
w["동추가 구간"] = pd.cut(w["동추가"], [-1, -0.02, -0.005, 0.005, 0.02, 1])
w.groupby("동추가 구간", observed=True).agg(n=("r", "size"), 동추가평균=("동추가", "mean"), r평균=("r", "mean"), 모델MAPE=("ape", "mean")).round(4)

C:\Users\UK\AppData\Local\Temp\ipykernel_203184\164342904.py:3: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  ref["사용가능일"] = ref["계약일자"] + pd.Timedelta(days=30)


직전 거래 재현: v2_trade 직전거래_층과 206,129건 모두 일치


,대상,n,직전 시세보정가,× 층비,× 동층비,모델 OOF
0,전체,114286,4.976,5.063,5.039,4.272
1,동이 달라 동층비 ≠ 층비,28798,5.113,5.299,5.204,4.337


상관(r, 동추가) 0.099


,n,동추가평균,r평균,모델MAPE
동추가 구간,,,,
"(-1.0, -0.02]",7175,-0.0405,0.0050,4.0674
"(-0.02, -0.005]",7320,-0.0120,0.0153,4.1675
"(-0.005, 0.005]",85488,-0.0000,0.0161,4.2496
"(0.005, 0.02]",7317,0.0118,0.0238,4.2893
"(0.02, 1.0]",6986,0.0402,0.0325,4.8417


## 6. 오차 20% 초과 거래

2023~25 중개거래 OOF에서 APE > 20%인 거래가 어떤 조건에 몰려 있는지 (조건별 비율: 전체 / 20% 초과 / 나머지).

In [7]:
d = trade.merge(oof, on="transaction_id", how="inner", validate="one_to_one")
d = d[d["거래유형"].eq("중개거래") & d["연"].between(2023, 2025)].copy()
d["등기일"] = pd.to_datetime(d["등기일자"].astype(str).str.strip(), format="%y.%m.%d", errors="coerce")
big = d["ape"] > 20
print(f"20% 초과 {big.sum():,}건 ({big.mean():.2%}), 오차 비중 {d.loc[big, 'ape'].sum() / d['ape'].sum():.1%} | "
      f"실제가 예측보다 낮음: 20% 초과 {(d.loc[big, 'r'] < 0).mean():.1%} vs 전체 {(d['r'] < 0).mean():.1%}")
same_day = d.groupby(["단지일련번호", "계약일자"])["transaction_id"].transform("size")
conds = {
    "직전 시세보정가와 ±20% 넘게 차이": np.log(d["거래금액"] / d["직전거래가_시세보정"]).abs() > np.log(1.2),
    "100세대 미만": d["세대수"] < 100,
    "동명 없음": d["동키"].isna(),
    "직전 거래 1년 넘음": d["직전거래_경과일"] > 365,
    "같은 면적 직전 거래 없음": d["직전거래가_동일면적"].isna(),
    "1~2층": d["층"] <= 2,
    "신규 단지": d["단지"].ne("기존"),
    "연식 35년 이상": d["연식"] >= 35,
    "전용 40㎡ 미만": d["전용면적"] < 40,
    "최고층": d["층"].eq(d["최고층"]),
    "같은 단지 · 같은 날 3건 이상": same_day >= 3,
    "매도자 개인 아님": d["매도자"].notna() & d["매도자"].ne("개인"),
    "매수자 개인 아님": d["매수자"].notna() & d["매수자"].ne("개인"),
}
pd.DataFrame([{"조건": k, "전체 %": v.mean() * 100, "20% 초과 %": v[big].mean() * 100, "나머지 %": v[~big].mean() * 100,
               "이 조건 MAPE": d.loc[v, "ape"].mean(), "n": int(v.sum())} for k, v in conds.items()]).round(2)

20% 초과 2,310건 (1.46%), 오차 비중 9.0% | 실제가 예측보다 낮음: 20% 초과 66.0% vs 전체 37.8%


,조건,전체 %,20% 초과 %,나머지 %,이 조건 MAPE,n
0,직전 시세보정가와 ±20% 넘게 차이,3.44,80.27,2.72,15.12,5160
1,100세대 미만,7.06,38.44,6.60,8.26,11196
2,동명 없음,7.35,34.81,6.95,7.75,11656
3,직전 거래 1년 넘음,9.18,30.69,8.86,7.45,14553
4,같은 면적 직전 거래 없음,2.23,22.99,1.93,11.34,3541
5,1~2층,10.60,18.96,10.47,5.51,16801
6,신규 단지,1.58,16.28,1.36,11.23,2500
7,연식 35년 이상,14.11,18.15,14.05,5.38,22367
8,전용 40㎡ 미만,6.11,12.77,6.01,5.83,9689
9,최고층,3.60,9.22,3.52,5.97,5714


In [8]:
cols = ["단지명", "시군구명", "계약일자", "전용면적", "층", "최고층", "세대수", "거래금액", "직전거래가_시세보정", "ape", "r", "단지"]
print("20% 초과 · 실제가 낮은 쪽 표본")
display(d[big & (d["r"] < 0)].sample(12, random_state=0)[cols].round(2))
print("20% 초과 · 실제가 높은 쪽 표본")
d[big & (d["r"] > 0)].sample(8, random_state=0)[cols].round(2)

20% 초과 · 실제가 낮은 쪽 표본


,단지명,시군구명,계약일자,전용면적,층,최고층,세대수,거래금액,직전거래가_시세보정,ape,r,단지
194001,동아한가람1,송파구,2024-03-09,84.38,17,18.0,782.0,88000,102018.31,27.54,-0.24,기존
73638,상계주공14(고층),노원구,2025-10-18,41.30,7,15.0,1635.0,28000,41201.75,41.76,-0.35,기존
486,대성아파트,종로구,2024-02-28,94.94,6,9.0,79.0,30000,42882.89,77.42,-0.57,기존
56652,한양수자인,도봉구,2023-06-27,84.99,15,15.0,80.0,43000,NaN,34.44,-0.30,기존
66097,포레나노원,노원구,2024-03-26,59.99,14,30.0,1062.0,72000,89770.89,23.12,-0.21,기존
199279,가락삼익맨숀,송파구,2025-03-19,84.83,7,12.0,936.0,137000,175085.90,21.01,-0.19,기존
93081,홍연,서대문구,2025-04-11,52.23,4,5.0,15.0,31300,45130.46,36.00,-0.31,기존
16686,옥수파크힐스117동~125동,성동구,2025-09-27,84.30,16,17.0,758.0,195000,243116.12,26.25,-0.23,기존
73762,한신동성,노원구,2025-10-11,138.29,11,15.0,498.0,90000,108348.53,20.49,-0.19,기존
153719,보라매파크빌,동작구,2023-06-23,84.73,3,15.0,423.0,68000,99079.36,48.38,-0.39,기존


20% 초과 · 실제가 높은 쪽 표본


,단지명,시군구명,계약일자,전용면적,층,최고층,세대수,거래금액,직전거래가_시세보정,ape,r,단지
153251,래미안트윈파크,동작구,2023-02-22,59.90,26,29.0,523.0,136000,129062.76,21.16,0.24,기존
8270,라이온즈,용산구,2025-10-17,121.90,3,5.0,14.0,175000,NaN,27.16,0.32,신규
180337,알파임하우스1,강남구,2023-08-02,242.16,9,10.0,19.0,550000,NaN,22.27,0.25,신규
90163,홍연,서대문구,2024-01-22,52.59,2,5.0,15.0,42000,NaN,26.76,0.31,기존
86209,제이타워4,은평구,2025-11-10,12.16,9,12.0,28.0,12000,10057.71,25.04,0.29,기존
2125,청구e편한세상,중구,2023-02-18,118.31,8,15.0,737.0,180000,157045.16,20.96,0.24,기존
94656,홍연,서대문구,2025-10-27,62.56,1,5.0,15.0,51500,29117.58,29.18,0.35,기존
180900,네오빌,강남구,2024-01-31,64.29,4,6.0,14.0,210000,NaN,64.74,1.04,신규


## 7. 등기 · 해제

- 실거래 `등기일자`는 2023년 계약부터 채워진다. 원본의 가장 늦은 등기일로 수집 시점을 추정하고, 등기가 충분히 끝났을 계약(2025-06 이전)에서 등기 없는 거래를 본다.
- 해제 거래는 V1 01에서 이미 뺐다 (위 `해제여부` 확인).

In [9]:
t2 = trade.copy()
t2["등기일"] = pd.to_datetime(t2["등기일자"].astype(str).str.strip(), format="%y.%m.%d", errors="coerce")
broker = t2["거래유형"].eq("중개거래")
print("등기일 최대 (원본 수집 시점 추정):", t2["등기일"].max().date())
print("계약 → 등기 소요일 10/50/90/99%:", np.round((t2.loc[broker, "등기일"] - t2.loc[broker, "계약일자"]).dt.days.quantile([.1, .5, .9, .99]).to_numpy(), 0))
by_q = t2[broker].groupby(t2.loc[broker, "계약일자"].dt.to_period("Q"))["등기일"].apply(lambda s: s.notna().mean()).round(3)
display(by_q.to_frame("등기 있음").T)
old = d[d["계약일자"] < "2025-07-01"].copy()
old["등기있음"] = old["등기일"].notna()
old.groupby("등기있음").agg(n=("ape", "size"), MAPE=("ape", "mean"), 초과20비율=("ape", lambda s: (s > 20).mean() * 100), r평균=("r", "mean")).round(3)

등기일 최대 (원본 수집 시점 추정):

 2026-08-19
계약 → 등기 소요일 10/50/90/99%: [ 37.  82. 126. 214.]


계약일자,2021Q4,2022Q1,2022Q2,2022Q3,2022Q4,2023Q1,2023Q2,2023Q3,2023Q4,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2
등기 있음,0.0,0.0,0.0,0.0,0.0,1.0,0.999,0.999,0.999,0.998,0.998,0.998,0.998,0.996,0.996,0.987,0.989,0.987,0.542


,n,MAPE,초과20비율,r평균
등기있음,,,,
False,308,6.252,4.545,0.031
True,126022,4.827,1.426,0.015


## 8. 정리

| 항목 | 결과 | 다음 |
|---|---|---|
| 동 · 호별 공시가격 | 2023~ 거래의 92~95% 연결. 동 프리미엄 80%가 0, 잔차 보정 0.014%p · 직전 거래가 보정은 악화 | 회귀 시험 후보 제외 (예상 0.01~0.05%p < 기준 0.1%p) |
| 오차 20% 초과 거래 | 비교 거래에서 크게 벗어난 거래, 소규모 · 비교 거래 희소 · 신규 단지에 집중. 데이터 오류 유형 없음 | 학습 · 평가 그대로. 소규모 단지(MAPE 8.26%)의 오차 범위 포함률은 V4 화면 설계 때 점검 |
| 등기 · 해제 | 등기 없는 거래 0.24%, 해제는 V1 01에서 제외 | 처리 없음 |

같은 단지 안 집 차이(향 · 조망 · 수리 상태)는 공개 데이터의 공시가격에도 거의 담겨 있지 않다. 남은 오차를 줄일 정보는 거래 시점의 호가(V4 사용자 입력)처럼 공개 데이터 밖에 있다.
V3 다음 단계는 EDA 2(신규 단지: 분양권 · 입주권 실거래, 분양가)와 EDA 3(단지 단위 가격 변화)다.